**为什么使用 TFRecord?**

正常情况下我们训练文件数据集经常会生成 train, test 或者 val 文件夹，这些文件夹内部往往会存着成千上万的图片或文本等文件，这些文件被**散列存着，这样不仅占用磁盘空间，并且再被一个个读取的时候会非常慢，繁琐**。占用**大量内存**空间（有的大型数据不足以一次性加载）。此时我们 TFRecord 格式的文件存储形式会很合理的帮我们存储数据。TFRecord 内部使用了 **“ProtocolBuffer”** 二进制数据编码方案，它只占用一个内存块，只需要一次性加载一个二进制文件的方式即可，简单，快速，尤其对大型训练数据很友好。而且当我们的训练数据量比较大的时候，可以将数据分成多个 TFRecord 文件，来提高处理效率。

In [17]:
!python3 --version

Python 3.11.11


In [18]:
!nvidia-smi

Tue Jun 10 12:14:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
import tensorflow as tf

from tensorflow import keras

print(tf.__version__)
print(sys.version_info)
for module in mpl, np, pd, sklearn, tf, keras:
    print(module.__name__, module.__version__)

2.18.0
sys.version_info(major=3, minor=11, micro=11, releaselevel='final', serial=0)
matplotlib 3.7.2
numpy 1.26.4
pandas 2.2.3
sklearn 1.2.2
tensorflow 2.18.0
keras._tf_keras.keras 3.8.0


In [20]:
# tfrecord 文件格式---往下层层分类
# -> tf.train.Example
#    -> tf.train.Features -> {"key": tf.train.Feature}
#       -> tf.train.Feature -> tf.train.ByteList/FloatList/Int64List

#学习时，从低往上练习
favorite_books = [name.encode('utf-8')
                  for name in ["machine learning", "cc150"]]
print(favorite_books)
favorite_books_bytelist = tf.train.BytesList(value = favorite_books) # 字符串用 BytesList
print(type(favorite_books_bytelist))
print(favorite_books_bytelist)

[b'machine learning', b'cc150']
<class 'tensorflow.core.example.feature_pb2.BytesList'>
value: "machine learning"
value: "cc150"



In [21]:
#hours设置到0-24之间
hours_floatlist = tf.train.FloatList(value = [15.5, 9.5, 7.0, 8.0])
print(type(hours_floatlist))
print(hours_floatlist)

<class 'tensorflow.core.example.feature_pb2.FloatList'>
value: 15.5
value: 9.5
value: 7.0
value: 8.0



In [22]:
age_int64list = tf.train.Int64List(value = [42])
print(type(age_int64list))
print(age_int64list)

<class 'tensorflow.core.example.feature_pb2.Int64List'>
value: 42



In [23]:
#进一步，开搞features
features = tf.train.Features(
    feature = {
        "favorite_books": tf.train.Feature(bytes_list = favorite_books_bytelist),
        "hours": tf.train.Feature(float_list = hours_floatlist),
        "age": tf.train.Feature(int64_list = age_int64list),
    }
)
print(type(features))
print(features) #类似于json的格式

<class 'tensorflow.core.example.feature_pb2.Features'>
feature {
  key: "age"
  value {
    int64_list {
      value: 42
    }
  }
}
feature {
  key: "favorite_books"
  value {
    bytes_list {
      value: "machine learning"
      value: "cc150"
    }
  }
}
feature {
  key: "hours"
  value {
    float_list {
      value: 15.5
      value: 9.5
      value: 7.0
      value: 8.0
    }
  }
}



In [24]:
#example又在外面加了features封装
example = tf.train.Example(features=features)
print(type(example))
print(example)

<class 'tensorflow.core.example.example_pb2.Example'>
features {
  feature {
    key: "age"
    value {
      int64_list {
        value: 42
      }
    }
  }
  feature {
    key: "favorite_books"
    value {
      bytes_list {
        value: "machine learning"
        value: "cc150"
      }
    }
  }
  feature {
    key: "hours"
    value {
      float_list {
        value: 15.5
        value: 9.5
        value: 7.0
        value: 8.0
      }
    }
  }
}



In [25]:
#需要把Example对象进行序列化后，才能写入文件
serialized_example = example.SerializeToString() 
print(type(serialized_example))
print(serialized_example)           # 序列化后变成字节流
print(len(serialized_example))

<class 'bytes'>
b'\n\\\n-\n\x0efavorite_books\x12\x1b\n\x19\n\x10machine learning\n\x05cc150\n\x1d\n\x05hours\x12\x14\x12\x12\n\x10\x00\x00xA\x00\x00\x18A\x00\x00\xe0@\x00\x00\x00A\n\x0c\n\x03age\x12\x05\x1a\x03\n\x01*'
94


In [26]:
!pwd;ls -al

/kaggle/working
total 16
drwxr-xr-x 4 root root 4096 Jun 10 12:13 .
drwxr-xr-x 5 root root 4096 Jun 10 12:11 ..
drwxr-xr-x 2 root root 4096 Jun 10 12:13 tfrecord_basic
drwxr-xr-x 2 root root 4096 Jun 10 12:11 .virtual_documents


In [27]:
!rm -rf ./tfrecord_basic

In [28]:
#生成test.tfrecords文件
output_dir = 'tfrecord_basic'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
filename = "test.tfrecords"
filename_fullpath = os.path.join(output_dir, filename)
with tf.io.TFRecordWriter(filename_fullpath) as writer:
    #把serialized_example写3遍到test.tfrecords里边
    for i in range(3):
        writer.write(serialized_example)

In [29]:
!pwd;ls -al;cd ./tfrecord_basic;ls -al;cat test.tfrecords

/kaggle/working
total 16
drwxr-xr-x 4 root root 4096 Jun 10 12:14 .
drwxr-xr-x 5 root root 4096 Jun 10 12:11 ..
drwxr-xr-x 2 root root 4096 Jun 10 12:14 tfrecord_basic
drwxr-xr-x 2 root root 4096 Jun 10 12:11 .virtual_documents
total 12
drwxr-xr-x 2 root root 4096 Jun 10 12:14 .
drwxr-xr-x 4 root root 4096 Jun 10 12:14 ..
-rw-r--r-- 1 root root  330 Jun 10 12:14 test.tfrecords
^       ذn�
\
-
favorite_books

machine learning
cc150

hours
  xA  A  �@   A

age
*dx�C^       ذn�
\
-
favorite_books

machine learning
cc150

hours
  xA  A  �@   A

age
*dx�C^       ذn�
\
-
favorite_books

machine learning
cc150

hours
  xA  A  �@   A

age
*dx�C

In [30]:
#读取record并打印
dataset = tf.data.TFRecordDataset([filename_fullpath])
for serialized_example_tensor in dataset:
    print(serialized_example_tensor)

tf.Tensor(b'\n\\\n-\n\x0efavorite_books\x12\x1b\n\x19\n\x10machine learning\n\x05cc150\n\x1d\n\x05hours\x12\x14\x12\x12\n\x10\x00\x00xA\x00\x00\x18A\x00\x00\xe0@\x00\x00\x00A\n\x0c\n\x03age\x12\x05\x1a\x03\n\x01*', shape=(), dtype=string)
tf.Tensor(b'\n\\\n-\n\x0efavorite_books\x12\x1b\n\x19\n\x10machine learning\n\x05cc150\n\x1d\n\x05hours\x12\x14\x12\x12\n\x10\x00\x00xA\x00\x00\x18A\x00\x00\xe0@\x00\x00\x00A\n\x0c\n\x03age\x12\x05\x1a\x03\n\x01*', shape=(), dtype=string)
tf.Tensor(b'\n\\\n-\n\x0efavorite_books\x12\x1b\n\x19\n\x10machine learning\n\x05cc150\n\x1d\n\x05hours\x12\x14\x12\x12\n\x10\x00\x00xA\x00\x00\x18A\x00\x00\xe0@\x00\x00\x00A\n\x0c\n\x03age\x12\x05\x1a\x03\n\x01*', shape=(), dtype=string)


I0000 00:00:1749557686.619002      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1749557686.619698      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [31]:
# VarLenFeature是变长的，得到的是sparseTensor,要通过to_dense变为Tensor，
# 如果FixedLenFeature，得到的是Tensor，必须传入原来保存时数据的shape
expected_features = {
    "favorite_books": tf.io.FixedLenFeature([2],dtype = tf.string),
#     "hours": tf.io.FixedLenFeature([4],dtype = tf.float32),
    "hours": tf.io.VarLenFeature(dtype = tf.float32),
    "age": tf.io.FixedLenFeature([], dtype = tf.int64),
}
dataset = tf.data.TFRecordDataset([filename_fullpath])
#sparse tensor 存储稀疏矩阵的时候效率比较高
for serialized_example_tensor in dataset:
    example = tf.io.parse_single_example(
        serialized_example_tensor, # 游标
        expected_features          # 解析的类型
    )
    print(example)
    #把books从sparse tensor解析出来
#     books = tf.sparse.to_dense(example["favorite_books"],
#                                default_value=b"")
#     print(books)
#     #这里是为了把两个字符串解析出来
#     for book in books:
#         print(book.numpy().decode("UTF-8"))
    for i in example["favorite_books"]:
        print(i.numpy().decode("UTF-8"))
    print('-'*50)
    hours = tf.sparse.to_dense(example["hours"])
    print(hours)
    for hour in hours:
        print(hour.numpy())
    print('-'*50)
    print(example["age"].numpy())
    

{'hours': SparseTensor(indices=tf.Tensor(
[[0]
 [1]
 [2]
 [3]], shape=(4, 1), dtype=int64), values=tf.Tensor([15.5  9.5  7.   8. ], shape=(4,), dtype=float32), dense_shape=tf.Tensor([4], shape=(1,), dtype=int64)), 'age': <tf.Tensor: shape=(), dtype=int64, numpy=42>, 'favorite_books': <tf.Tensor: shape=(2,), dtype=string, numpy=array([b'machine learning', b'cc150'], dtype=object)>}
machine learning
cc150
--------------------------------------------------
tf.Tensor([15.5  9.5  7.   8. ], shape=(4,), dtype=float32)
15.5
9.5
7.0
8.0
--------------------------------------------------
42
{'hours': SparseTensor(indices=tf.Tensor(
[[0]
 [1]
 [2]
 [3]], shape=(4, 1), dtype=int64), values=tf.Tensor([15.5  9.5  7.   8. ], shape=(4,), dtype=float32), dense_shape=tf.Tensor([4], shape=(1,), dtype=int64)), 'age': <tf.Tensor: shape=(), dtype=int64, numpy=42>, 'favorite_books': <tf.Tensor: shape=(2,), dtype=string, numpy=array([b'machine learning', b'cc150'], dtype=object)>}
machine learning
cc150
----

In [32]:
#把tfrecord存为压缩文件
filename_fullpath_zip = filename_fullpath + '.zip'
options = tf.io.TFRecordOptions(compression_type = "GZIP")
with tf.io.TFRecordWriter(filename_fullpath_zip, options) as writer:
    for i in range(3):
        writer.write(serialized_example)

In [33]:
!ls -l tfrecord_basic

total 8
-rw-r--r-- 1 root root 330 Jun 10 12:14 test.tfrecords
-rw-r--r-- 1 root root 127 Jun 10 12:14 test.tfrecords.zip


In [34]:
#压缩后的文件的读取方法
dataset_zip = tf.data.TFRecordDataset([filename_fullpath_zip], 
                                      compression_type= "GZIP")
for serialized_example_tensor in dataset_zip:
    example = tf.io.parse_single_example(
        serialized_example_tensor,
        expected_features)
    print(example)

{'hours': SparseTensor(indices=tf.Tensor(
[[0]
 [1]
 [2]
 [3]], shape=(4, 1), dtype=int64), values=tf.Tensor([15.5  9.5  7.   8. ], shape=(4,), dtype=float32), dense_shape=tf.Tensor([4], shape=(1,), dtype=int64)), 'age': <tf.Tensor: shape=(), dtype=int64, numpy=42>, 'favorite_books': <tf.Tensor: shape=(2,), dtype=string, numpy=array([b'machine learning', b'cc150'], dtype=object)>}
{'hours': SparseTensor(indices=tf.Tensor(
[[0]
 [1]
 [2]
 [3]], shape=(4, 1), dtype=int64), values=tf.Tensor([15.5  9.5  7.   8. ], shape=(4,), dtype=float32), dense_shape=tf.Tensor([4], shape=(1,), dtype=int64)), 'age': <tf.Tensor: shape=(), dtype=int64, numpy=42>, 'favorite_books': <tf.Tensor: shape=(2,), dtype=string, numpy=array([b'machine learning', b'cc150'], dtype=object)>}
{'hours': SparseTensor(indices=tf.Tensor(
[[0]
 [1]
 [2]
 [3]], shape=(4, 1), dtype=int64), values=tf.Tensor([15.5  9.5  7.   8. ], shape=(4,), dtype=float32), dense_shape=tf.Tensor([4], shape=(1,), dtype=int64)), 'age': <tf.Tensor: